# SAM Segmentasyon Notebook'u / SAM Segmentation Notebook

Bu notebook, Segment Anything Model (SAM) kullanarak instance segmentasyonunun nasıl yapılacağını gösterir.
This notebook demonstrates how to use the Segment Anything Model (SAM) for instance segmentation.

## SAM Nedir? / What is SAM?

Facebook AI Research tarafından geliştirilen SAM, herhangi bir nesneyi segmentlere ayırabilen güçlü bir segmentasyon modelidir.
SAM (Segment Anything Model) is a powerful segmentation model developed by Facebook AI Research that can segment any object.

### Özellikleri / Features:
- **Prompt-based**: Nokta, kutu veya metin ile nesne seçimi
- **Zero-shot**: Eğitim verisi olmadan yeni nesneleri segment edebilir
- **Real-time**: Gerçek zamanlı segmentasyon desteği
- **Instance Segmentation**: Her nesne için ayrı maske üretir

In [ ]:
# Gerekli kütüphaneleri içe aktar / Import required libraries
import os
import sys
from pathlib import Path

# Proje kök dizinini path'e ekle / Add project root to path
project_root = Path('../').resolve()
sys.path.insert(0, str(project_root))

import cv2
import numpy as np
from PIL import Image
import matplotlib.pyplot as plt
import torch

print("Kütüphaneler başarıyla içe aktarıldı!")
print(f"PyTorch sürümü: {torch.__version__}")
print(f"CUDA mevcut: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"CUDA cihazı: {torch.cuda.get_device_name(0)}")

## SAM Model Yükleme / Loading SAM Model

SAM modelini HuggingFace Transformers kütüphanesinden yüklüyoruz.
We load the SAM model from HuggingFace Transformers library.

In [ ]:
# HuggingFace Transformers'dan SAM yükle / Load SAM from HuggingFace Transformers
from transformers import SamProcessor, SamModel

# Model seçenekleri / Model options:
# - facebook/sam-vit-base (temel - orta boyut)
# - facebook/sam-vit-large (büyük)
# - facebook/sam-vit-huge (en büyük)
# - facebook/sam2-base (SAM 2 temel)
# - facebook/sam2-large (SAM 2 büyük)

MODEL_ID = "facebook/sam-vit-base"
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

print(f"SAM modeli yükleniyor: {MODEL_ID}")
print(f"Cihaz: {DEVICE}")

# Processor ve Model yükle / Load processor and model
processor = SamProcessor.from_pretrained(MODEL_ID)
model = SamModel.from_pretrained(MODEL_ID)

# Modeli cihaza taşı / Move model to device
model.to(DEVICE)
model.eval()

print("SAM modeli başarıyla yüklendi!")

## Örnek Görüntü Yükleme / Loading Sample Image

Test için örnek bir görüntü yüklüyoruz. Gerçek uygulamada defekt tespiti için madencilik görüntüleri kullanılabilir.
We load a sample image for testing. In real applications, mining images can be used for defect detection.

In [ ]:
# Örnek görüntü oluştur veya yükle / Create or load sample image
# Burada test için sentetik bir görüntü oluşturuyoruz
# In production, load actual mining/conveyor belt images

def create_test_image():
    """Test görüntüsü oluştur / Create test image"""
    img = np.zeros((512, 512, 3), dtype=np.uint8)
    
    # Arka plan / Background
    img[:] = (100, 100, 100)
    
    # Nesne 1: Daire (cevher parçası) / Circle (ore piece)
    cv2.circle(img, (150, 150), 60, (180, 80, 50), -1)
    cv2.circle(img, (150, 150), 60, (200, 100, 60), 3)
    
    # Nesne 2: Dikdörtgen (defekt bölgesi) / Rectangle (defect area)
    cv2.rectangle(img, (300, 100), (450, 250), (80, 60, 120), -1)
    cv2.rectangle(img, (300, 100), (450, 250), (100, 80, 140), 3)
    
    # Nesne 3: Elips (anomali) / Ellipse (anomaly)
    cv2.ellipse(img, (200, 350), (80, 50), 30, 0, 360, (120, 100, 80), -1)
    cv2.ellipse(img, (200, 350), (80, 50), 30, 0, 360, (140, 120, 100), 3)
    
    # Nesne 4: Üçgen (yabancı madde) / Triangle (foreign material)
    pts = np.array([[400, 320], [450, 420], [350, 420]], np.int32)
    cv2.fillPoly(img, [pts], (90, 130, 150))
    cv2.polylines(img, [pts], True, (110, 150, 170), 3)
    
    return img

test_image = create_test_image()
image_pil = Image.fromarray(test_image)

# Görüntüyü göster / Display image
plt.figure(figsize=(10, 10))
plt.imshow(test_image)
plt.title("Test Görüntüsü / Test Image")
plt.axis('off')
plt.show()

print(f"Görüntü boyutu: {test_image.shape}")

## Nokta ile Segmentasyon / Segmentation with Points

SAM, kullanıcının verdiği noktalardan nesneleri segment edebilir. Burada iki tür nokta kullanılır:
SAM can segment objects from points provided by the user. Two types of points are used here:
- **Foreground (1)**: Nesnenin içindeki nokta
- **Background (0)**: Nesnenin dışındaki nokta

In [ ]:
# Nokta tabanlı segmentasyon / Point-based segmentation

# Her nesne için foreground noktaları / Foreground points for each object
input_points = np.array([
    [150, 150],  # Daire / Circle
    [375, 175],  # Dikdörtgen / Rectangle
    [200, 350],  # Elips / Ellipse
    [400, 370],  # Üçgen / Triangle
])

input_labels = np.array([1, 1, 1, 1])  # Hepsi foreground

# Görüntüyü işle / Process image
inputs = processor(
    image_pil,
    input_points=input_points,
    input_labels=input_labels,
    return_tensors="pt"
).to(DEVICE)

# Inference / Inference
with torch.no_grad():
    outputs = model(**inputs)

# Maskeleri işle / Post-process masks
masks = processor.image_processor.post_process_masks(
    outputs.pred_masks,
    inputs["original_sizes"],
    inputs["reshaped_input_sizes"]
)

# En iyi maskeleri seç / Select best masks
masks = masks[0].cpu().numpy()

print(f"Toplam maske sayısı: {len(masks)}")
print(f"Maske şekli / Mask shape: {masks.shape}")

## Sonuçları Görselleştirme / Visualizing Results

Segmentasyon sonuçlarını görselleştiriyoruz.
We visualize the segmentation results.

In [ ]:
# Sonuçları görselleştir / Visualize results

def visualize_sam_results(image, masks, input_points, input_labels):
    """SAM sonuçlarını görselleştir / Visualize SAM results"""
    fig, axes = plt.subplots(1, 3, figsize=(18, 6))
    
    # Orijinal görüntü / Original image
    axes[0].imshow(image)
    axes[0].set_title("Orijinal Görüntü / Original Image")
    axes[0].axis('off')
    
    # Noktaları göster / Show points
    img_with_points = image.copy()
    for i, (point, label) in enumerate(zip(input_points, input_labels)):
        color = (0, 255, 0) if label == 1 else (255, 0, 0)  # Yeşil: foreground, Kırmızı: background
        cv2.circle(img_with_points, tuple(point), 10, color, -1)
        cv2.circle(img_with_points, tuple(point), 12, (255, 255, 255), 2)
    axes[1].imshow(img_with_points)
    axes[1].set_title("Girdi Noktaları / Input Points")
    axes[1].axis('off')
    
    # Maskeleri göster / Show masks
    result_img = image.copy()
    colors = [
        (255, 0, 0),    # Kırmızı / Red
        (0, 255, 0),    # Yeşil / Green
        (0, 0, 255),    # Mavi / Blue
        (255, 255, 0),  # Sarı / Yellow
    ]
    
    for i, mask in enumerate(masks):
        if mask.sum() > 0:  # Geçerli maske
            color = colors[i % len(colors)]
            # Maskeyi renklendir / Color the mask
            mask_bool = mask > 0
            result_img[mask_bool] = (
                result_img[mask_bool] * 0.5 + 
                np.array(color) * 0.5
            ).astype(np.uint8)
    
    axes[2].imshow(result_img)
    axes[2].set_title("Segmentasyon Sonuçları / Segmentation Results")
    axes[2].axis('off')
    
    plt.tight_layout()
    plt.show()
    
    return result_img

# Görselleştir / Visualize
result = visualize_sam_results(test_image, masks, input_points, input_labels)

## Otomatik Maske Üretimi / Automatic Mask Generation

SAM, herhangi bir prompt olmadan görüntüdeki tüm nesneleri otomatik olarak segment edebilir.
SAM can automatically segment all objects in an image without any prompt.

In [ ]:
# Otomatik maske üretimi / Automatic mask generation

def generate_automatic_masks(image, num_points=32):
    """
    Görüntüden otomatik maskeler üretir.
    Generates automatic masks from an image.
    
    Args:
        image: PIL Image
        num_points: Izgara boyutu (num_points x num_points)
    
    Returns:
        masks: Segmentasyon maskeleri
    """
    w, h = image.size
    
    # Izgara noktaları oluştur / Create grid points
    grid_size = int(np.sqrt(num_points))
    points = []
    labels = []
    
    for i in range(grid_size):
        for j in range(grid_size):
            x = int((i + 0.5) * w / grid_size)
            y = int((j + 0.5) * h / grid_size)
            points.append([x, y])
            labels.append(1)  # Hepsi foreground
    
    points = np.array(points)
    labels = np.array(labels)
    
    # Modeli çağır / Call model
    inputs = processor(
        image,
        input_points=points,
        input_labels=labels,
        return_tensors="pt"
).to(DEVICE)
    
    with torch.no_grad():
        outputs = model(**inputs)
    
    # Maskeleri işle / Post-process
    masks = processor.image_processor.post_process_masks(
        outputs.pred_masks,
        inputs["original_sizes"],
        inputs["reshaped_input_sizes"]
    )
    
    return masks[0].cpu().numpy()

# Otomatik maskeler üret / Generate automatic masks
print("Otomatik maskeler üretiliyor... / Generating automatic masks...")
auto_masks = generate_automatic_masks(image_pil, num_points=64)

print(f"Otomatik maske sayısı: {len(auto_masks)}")
print(f"Maske şekli / Mask shape: {auto_masks.shape}")

## Otomatik Sonuçları Görselleştirme / Visualizing Automatic Results

Otomatik olarak üretilen maskeleri görselleştiriyoruz.
We visualize the automatically generated masks.

In [ ]:
# Otomatik sonuçları görselleştir / Visualize automatic results

fig, axes = plt.subplots(2, 3, figsize=(15, 10))

# Orijinal görüntü / Original image
axes[0, 0].imshow(test_image)
axes[0, 0].set_title("Orijinal / Original")
axes[0, 0].axis('off')

# İlk birkaç maskeyi göster / Show first few masks
for idx in range(5):
    row = (idx + 1) // 3
    col = (idx + 1) % 3
    
    if idx < len(auto_masks):
        mask = auto_masks[idx]
        if mask.sum() > 0:
            # Maskeyi görselleştir / Visualize mask
            mask_vis = np.zeros_like(test_image)
            mask_vis[mask > 0] = [255, 255, 255]
            axes[row, col].imshow(mask_vis)
            axes[row, col].set_title(f"Maske {idx+1} / Mask {idx+1}")
        else:
            axes[row, col].imshow(test_image)
            axes[row, col].set_title(f"Boş / Empty {idx+1}")
    axes[row, col].axis('off')

plt.tight_layout()
plt.show()

# Tüm maskelerin birleşimi / Union of all masks
combined_mask = np.zeros_like(test_image)
for mask in auto_masks:
    if mask.sum() > 100:  # Eşik değeri
        combined_mask[mask > 0] = 255

plt.figure(figsize=(10, 10))
plt.imshow(combined_mask)
plt.title("Tüm Maskelerin Birleşimi / Union of All Masks")
plt.axis('off')
plt.show()

## Endüstriyel Uygulama: Defekt Tespiti / Industrial Application: Defect Detection

SAM'ı madencilik endüstrisinde defekt tespiti için nasıl kullanabileceğimizi gösteriyoruz.
We demonstrate how SAM can be used for defect detection in the mining industry.

In [ ]:
# Endüstriyel uygulama: Defekt tespiti / Industrial application: Defect detection

def detect_industrial_defects(image, defect_points):
    """
    Endüstriyel defektleri tespit eder.
    Detects industrial defects.
    
    Args:
        image: PIL Image
        defect_points: Defekt bölgelerinin koordinatları
    
    Returns:
        results: Tespit edilen defektler
    """
    # Noktaları ve etiketleri hazırla / Prepare points and labels
    input_points = np.array(defect_points)
    input_labels = np.ones(len(defect_points), dtype=np.int64)
    
    # Modeli çağır / Call model
    inputs = processor(
        image,
        input_points=input_points,
        input_labels=input_labels,
        return_tensors="pt"
).to(DEVICE)
    
    with torch.no_grad():
        outputs = model(**inputs)
    
    # Maskeleri işle / Post-process
    masks = processor.image_processor.post_process_masks(
        outputs.pred_masks,
        inputs["original_sizes"],
        inputs["reshaped_input_sizes"]
    )
    
    return masks[0].cpu().numpy()

# Örnek defekt noktaları / Sample defect points
defect_points = [
    [375, 175],   # Dikdörtgen defekt
    [200, 350],   # Elips anomali
    [400, 370],   # Üçgen yabancı madde
]

# Defektleri tespit et / Detect defects
defect_masks = detect_industrial_defects(image_pil, defect_points)

# Sonuçları görselleştir / Visualize results
fig, axes = plt.subplots(1, 2, figsize=(14, 7))

axes[0].imshow(test_image)
axes[0].set_title("Orijinal Görüntü / Original Image")
axes[0].axis('off')

# Defektleri göster / Show defects
defect_img = test_image.copy()
for i, mask in enumerate(defect_masks):
    if mask.sum() > 0:
        # Kırmızı ile defekt bölgelerini işaretle / Mark defect areas in red
        defect_img[mask > 0] = (
            defect_img[mask > 0] * 0.3 + 
            np.array([255, 0, 0]) * 0.7
        ).astype(np.uint8)

axes[1].imshow(defect_img)
axes[1].set_title("Tespit Edilen Defektler / Detected Defects")
axes[1].axis('off')

plt.tight_layout()
plt.show()

print(f"Tespit edilen defekt sayısı: {sum(1 for m in defect_masks if m.sum() > 0)}")

## Bounding Box ile Segmentasyon / Segmentation with Bounding Box

SAM, bounding box promptları ile de kullanılabilir.
SAM can also be used with bounding box prompts.

In [ ]:
# Bounding box ile segmentasyon / Segmentation with bounding box

def segment_with_box(image, boxes):
    """
    Bounding box ile segmentasyon yapar.
    Performs segmentation with bounding box.
    
    Args:
        image: PIL Image
        boxes: [[x1, y1, x2, y2], ...] formatında kutular
    
    Returns:
        masks: Segmentasyon maskeleri
    """
    all_masks = []
    
    for box in boxes:
        x1, y1, x2, y2 = box
        
        # Box'tan noktalar oluştur / Create points from box
        # Köşe noktaları ve merkez noktası / Corner points and center point
        points = np.array([
            [x1, y1],  # Sol üst / Top left
            [x2, y2],  # Sağ alt / Bottom right
            [(x1 + x2) // 2, (y1 + y2) // 2],  # Merkez / Center
        ])
        
        labels = np.array([1, 1, 1])  # Hepsi foreground
        
        inputs = processor(
            image,
            input_points=points,
            input_labels=labels,
            return_tensors="pt"
        ).to(DEVICE)
        
        with torch.no_grad():
            outputs = model(**inputs)
        
        masks = processor.image_processor.post_process_masks(
            outputs.pred_masks,
            inputs["original_sizes"],
            inputs["reshaped_input_sizes"]
        )
        
        all_masks.append(masks[0].cpu().numpy())
    
    return all_masks

# Örnek kutular / Sample boxes
boxes = [
    [100, 100, 200, 200],   # Daire bölgesi
    [300, 100, 450, 250],  # Dikdörtgen bölge
    [120, 300, 280, 400],  # Elips bölge
]

# Box ile segmentasyon yap / Segment with boxes
box_masks = segment_with_box(image_pil, boxes)

# Sonuçları görselleştir / Visualize results
result_img = test_image.copy()
colors = [(255, 0, 0), (0, 255, 0), (0, 0, 255)]

for i, masks in enumerate(box_masks):
    box = boxes[i]
    color = colors[i]
    
    # Kutuyu çiz / Draw box
    cv2.rectangle(result_img, (box[0], box[1]), (box[2], box[3]), color, 2)
    
    # Maskeyi uygula / Apply mask
    for mask in masks:
        if mask.sum() > 0:
            result_img[mask > 0] = (
                result_img[mask > 0] * 0.5 + 
                np.array(color) * 0.5
            ).astype(np.uint8)

plt.figure(figsize=(10, 10))
plt.imshow(result_img)
plt.title("Bounding Box ile Segmentasyon / Segmentation with Bounding Box")
plt.axis('off')
plt.show()

## Performans Karşılaştırması / Performance Comparison

Farklı segmentasyon yöntemlerinin performansını karşılaştırıyoruz.
We compare the performance of different segmentation methods.

In [ ]:
# Performans karşılaştırması / Performance comparison
import time

def measure_inference_time(model, processor, image, method, **kwargs):
    """Inference süresini ölçer / Measures inference time"""
    # Warmup
    _ = processor(image, return_tensors="pt").to(DEVICE)
    with torch.no_grad():
        _ = model(**processor(image, return_tensors="pt").to(DEVICE))
    
    # Gerçek ölçüm / Real measurement
    start_time = time.time()
    iterations = 10
    
    for _ in range(iterations):
        inputs = processor(image, return_tensors="pt").to(DEVICE)
        with torch.no_grad():
            outputs = model(**inputs)
    
    end_time = time.time()
    avg_time = (end_time - start_time) / iterations
    
    return avg_time

# Farklı izgara boyutları için süre ölç / Measure time for different grid sizes
grid_sizes = [4, 8, 16, 32]
times = []

for grid_size in grid_sizes:
    num_points = grid_size * grid_size
    
    w, h = image_pil.size
    points = []
    labels = []
    
    for i in range(grid_size):
        for j in range(grid_size):
            x = int((i + 0.5) * w / grid_size)
            y = int((j + 0.5) * h / grid_size)
            points.append([x, y])
            labels.append(1)
    
    points = np.array(points)
    labels = np.array(labels)
    
    # Warmup
    _ = processor(image_pil, input_points=points[:1], input_labels=labels[:1], return_tensors="pt").to(DEVICE)
    
    start_time = time.time()
    for _ in range(5):
        inputs = processor(image_pil, input_points=points, input_labels=labels, return_tensors="pt").to(DEVICE)
        with torch.no_grad():
            outputs = model(**inputs)
    end_time = time.time()
    
    avg_time = (end_time - start_time) / 5
    times.append(avg_time)
    print(f"Izgara boyutu {grid_size}x{grid_size} ({num_points} nokta): {avg_time:.3f} saniye")

# Grafik / Plot
plt.figure(figsize=(10, 6))
plt.plot([g*g for g in grid_sizes], times, 'bo-', linewidth=2, markersize=8)
plt.xlabel('Nokta Sayısı / Number of Points')
plt.ylabel('Ortalama Süre (s) / Average Time (s)')
plt.title('SAM Performans Karşılaştırması / SAM Performance Comparison')
plt.grid(True)
plt.show()

## Özet / Summary

Bu notebook'ta şunları öğrendik:
In this notebook we learned:

1. **SAM Model Yükleme**: HuggingFace Transformers'dan SAM modelini yükleme
2. **Nokta ile Segmentasyon**: Kullanıcı tarafından verilen noktalardan nesne segmentasyonu
3. **Otomatik Maske Üretimi**: Herhangi bir prompt olmadan otomatik segmentasyon
4. **Endüstriyel Uygulama**: Defekt tespiti için SAM kullanımı
5. **Bounding Box ile Segmentasyon**: Kutu promptları ile segmentasyon
6. **Performans Analizi**: Farklı konfigürasyonların performans karşılaştırması

### Sonraki Adımlar / Next Steps:
- DETR/YOLOS ile nesne tespiti kılavuzuna geçin
- Model karşılaştırması yapın
- Endüstriyel uygulamaları detaylandırın